# 02B Append All vs Append New Only Demo

Use one Bronze source, one Full Read and one transformed DataFrame to compare two independent Silver targets across three days.


### Before you begin

Run 00B and then 00C to stage Day 1 in Bronze. Run this notebook once for Day 1. For Day 2, set `DEMO_DAY = 2` and rerun; the short staging cell adds four new movements. Day 3 has no new source records. Do not rerun 00C after advancing the scenario.


# 0. Environment — Setup Notebook

Run `setup_notebook()` to load `00_env_config`, resolve the caller environment, validate targets, and display configuration readiness. For shared configuration, pass `config_workspace_id` here. Setup Pipeline reuses this configuration.

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    finalize_pipeline,
    orchestrate_read,
    orchestrate_write,
    read_lakehouse_csv,
    read_lakehouse_table,
    setup_notebook,
    setup_pipeline,
    write_lakehouse_table,
)


context = setup_notebook()

# For shared config, use the workspace hosting 00_env_config:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


## Demo day

00C prepares Day 1. After a successful run, change to `2` and rerun for the new batch, then `3` for the no-change run. Never return to Day 1 after advancing Bronze.


In [ ]:
DEMO_DAY = 1

if DEMO_DAY not in {1, 2, 3}:
    raise ValueError("DEMO_DAY must be 1, 2, or 3.")


## Prepare the Day 2 Bronze input

00C already staged Day 1. This is demo input preparation, not incremental pipeline logic. Day 3 changes nothing.

This is explicit demo source preparation before pipeline selection. It changes Bronze independently of the later pipeline execution; the orchestrated Reads and Writes use the selected mode. Run it deliberately when advancing the demo day.


In [ ]:
if DEMO_DAY == 2:
    day2_df = read_lakehouse_csv(
        "Demo/inventory_day2.csv", store="Bronze",
        header=True, inferSchema=True,
    )
    existing_df = read_lakehouse_table(
        table_name="inventory_movements", store="Bronze", schema="demo",
    )
    # Replaying Day 2 keeps each movement_id once in the Bronze source.
    updated_df = existing_df.unionByName(day2_df).dropDuplicates(["movement_id"])
    write_lakehouse_table(
        updated_df, table_name="inventory_movements", store="Bronze", schema="demo", mode="overwrite",
    )
    print("Bronze inventory rows:", updated_df.count())


# 1. Setup Pipeline

Use **Dry Run** to preview without writing, **Ungoverned** to run without Data Contracts, or **Governed** requires activated deployments and Contracts; it remains blocked until trusted deployment verification is integrated. Pipeline source participation and reporting are automatic.

Run the final Finalize cell after the complete pipeline. Dry Run returns a session-only summary; persistent modes record the actual outcome.


In [ ]:
PIPELINE_MODE = "Ungoverned"  # Dry Run, Ungoverned, Governed

pipeline = setup_pipeline(
    pipeline_mode=PIPELINE_MODE,
    observe_source=False,  # True requests full physical Source Observation this run.
    source_observation_days=None,  # Set e.g. 7 for automatic observation; None disables it.
)


# 2. Read

Read the complete inventory source each day: 8 rows on Day 1, then 12 on Days 2 and 3. Full Read needs no target checkpoint or watermark. Both Writes use this one Read and exactly the same transformed DataFrame.


In [ ]:
# Keep complete Read results for transformation and lineage.
sources = {}


In [ ]:
source = orchestrate_read(
    store="Bronze",
    schema="demo",
    table_name="inventory_movements",
    read_mode="full",
    verbose=True,
    dq_diagnostics="sample",
)
sources["inventory_movements"] = source

# Inspect the actual rows returned by this Read.
display(sources["inventory_movements"]["dataframe"].limit(10))


# 3. Transform


In [ ]:
transformed_df = (
    sources["inventory_movements"]["dataframe"]
    .withColumn("movement_date", F.to_date("modified_datetime"))
    .withColumn("quantity_in", F.greatest(F.col("quantity_change"), F.lit(0)))
    .withColumn("quantity_out", F.greatest(-F.col("quantity_change"), F.lit(0)))
)

# Optional development inspection.
# display(transformed_df)


# 4. Write

Both independent Writes receive `transformed_df` without another Read or transformation. Append All inserts every submitted row; Append New Only deduplicates incoming `_row_hash` values and compares them with the target using a distributed anti-join. Changed hashes become additional rows, including changes to an existing business key.

A new Full Read execution repeats Append All. Receipts protect exact publication retries, not new executions. If one Write succeeds and the other fails, publication is partial: inspect the results and recover the failed publication before starting another execution. Always run Finalize.


In [ ]:
# Keep complete Write results for optional inspection.
writes = {}


## WRITE 1 — Append All


In [ ]:
writes["inventory_movements_append_all"] = orchestrate_write(
    transformed_df,

    store="Silver",
    schema="demo",
    table_name="inventory_movements_append_all",
    write_mode="append_all",

    repartition_by=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)

# Inspect the DataFrame submitted to this Write.
display(writes["inventory_movements_append_all"]["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(writes["inventory_movements_append_all"].get("dq_result"))
# display((writes["inventory_movements_append_all"].get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the persisted target after a successful persistent execution.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="inventory_movements_append_all").limit(10))


## WRITE 2 — Append New Only


In [ ]:
writes["inventory_movements_append_new_only"] = orchestrate_write(
    transformed_df,

    store="Silver",
    schema="demo",
    table_name="inventory_movements_append_new_only",
    write_mode="append_new_only",

    repartition_by=None,

    verbose=True,
    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)

# Inspect the DataFrame submitted to this Write.
display(writes["inventory_movements_append_new_only"]["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(writes["inventory_movements_append_new_only"].get("dq_result"))
# display((writes["inventory_movements_append_new_only"].get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the persisted target after a successful persistent execution.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="inventory_movements_append_new_only").limit(10))


In [ ]:
# Inspect persisted totals after both independent Writes complete.
# The comparison table assumes one complete persistent execution per day.
if writes["inventory_movements_append_all"].get("published"):
    for target in ("inventory_movements_append_all", "inventory_movements_append_new_only"):
        saved = read_lakehouse_table(store="Silver", schema="demo", table_name=target)
        print(target, saved.count())
else:
    print("No publication in this execution; persisted totals are not checked (for example, Dry Run).")


## Expected results and repeat runs

Run exactly one complete persistent execution per day, in order, starting with fresh Silver targets. Dry Run evaluates input without publishing; it does not demonstrate target hash filtering or these persisted counts. Ungoverned cannot write an already governed target. Governed requires approved activations and remains blocked pending trusted deployment verification. Call `finalize_pipeline(pipeline)` after both Writes before advancing the day.

| Day | Source rows | Append All total | Append New Only total |
|---|---:|---:|---:|
| 1 | 8 | 8 | 8 |
| 2 | 12 | 20 | 12 |
| 3 | 12 | 32 | 12 |

Day 3 still reads and submits all 12 rows. Append All adds 12; Append New Only inserts none. It may recover an already verified publication receipt (`already_applied`) for the unchanged batch; a newly evaluated batch with no unseen hashes reports `no_unseen_hashes`, a nonpublishing no-op. Neither means an empty incremental scope. Inspect persisted Silver counts separately from the prepared-input previews.

### Choose the strategy

- **Incremental Read + Append All:** suitable for large sources with reliable watermarks or change tracking. Reads newly eligible records and avoids repeated complete processing scans. Correct checkpoints and replay handling are essential; replaying consumed records as new work can duplicate them. Full Source Observation or profiling, when due, can still scan the complete source.
- **Full Read + Append New Only:** suitable when duplicate prevention matters or reliable incremental extraction is unavailable. Reads the complete source every time and inserts only unseen row hashes. The cost is full source scanning and distributed hash comparison. Unchanged data causes no additional business rows.
- **Incremental Read + Append New Only:** valid when extraction can limit the scope and hash comparison should also prevent replay duplicates within that scope. It cannot discover records excluded by the watermark.

Hashes cover final values and logical types after transformation and Sensitive Data treatment. Source timestamps and Job IDs participate; FabricOps audit and SCD lifecycle fields are excluded. Avoid adding volatile values to this experiment. Append New Only does not update keys, infer deletions or remove existing duplicates. Use SCD1 for current truth or SCD2 for history.

For an independent replay, use a fresh Guided Demo environment or separate demo table identities and approved Contracts. Keep targets, publication markers and scoped `METADATA_PROCESSING_STATE` together; do not delete checkpoints to replay work. Do not rerun 00C after advancing Bronze. See the [Guided Demo](https://voycepeh.github.io/FabricOps-Starter-Kit/guided-demo/02B-build-and-run-incremental-append-etl/) for persisted-count inspection and recovery guidance.


# 5. Finalize

Complete the run after every Read, transformation and Write. Dry Run keeps its summary in the session only. Ungoverned and Governed record the real outcome; Governed also releases its activation reservation. A stopped notebook remains incomplete.

In [ ]:
# 5. Finalize
pipeline_result = finalize_pipeline(pipeline)